In [15]:
#!git clone https://github.com/Felix-Schwer/rainfall_kf.git
%cd /content/rainfall_kf
!git pull origin main
!pip uninstall -y rainfall_kf
!pip install -e .

/content/rainfall_kf
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 14 (delta 2), reused 14 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (14/14), 348.65 KiB | 6.58 MiB/s, done.
From https://github.com/Felix-Schwer/rainfall_kf
 * branch            main       -> FETCH_HEAD
   74e394f..460188d  main       -> origin/main
Updating 74e394f..460188d
Fast-forward
 data/HUC8 - CONUS Upper Merced.cpg   |    1 +
 data/HUC8 - CONUS Upper Merced.dbf   |  Bin 0 -> 1105 bytes
 data/HUC8 - CONUS Upper Merced.prj   |    1 +
 data/HUC8 - CONUS Upper Merced.qmd   |  113 ++++
 data/HUC8 - CONUS Upper Merced.shp   |  Bin 0 -> 220092 bytes
 data/HUC8 - CONUS Upper Merced.shx   |  Bin 0 -> 108 bytes
 data/HUC8 - CONUS Upper Tuolumne.cpg |    1 +
 data/HUC8 - CONUS Upper Tuolumne.dbf |  Bin 0 -> 1105 bytes
 data/HUC8 - CONUS Upper Tuolumne.prj |    1 +
 data/HUC8 - CONUS Upper Tuolumne.qmd |  113 

In [1]:
import xarray as xr
import geopandas as gpd

In [3]:
da = xr.open_dataset('/content/rainfall_kf/data/gpm_sjv_huc8_subset.nc')
db = xr.open_dataset('/content/rainfall_kf/data/gpm_sjv_subset.nc')

In [4]:
merceddf = gpd.read_file('/content/rainfall_kf/data/HUC8 - CONUS Upper Merced.shp')
merceddf.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path


def pick_data_var(ds):
    """Pick a likely gridded time-varying variable from a dataset."""
    candidates = []
    for name, da_var in ds.data_vars.items():
        dims_lower = {d.lower() for d in da_var.dims}
        if any(t in dims_lower for t in ["time", "valid_time"]):
            if len(da_var.dims) >= 3:
                candidates.append(name)
    if not candidates:
        candidates = list(ds.data_vars)
    if not candidates:
        raise ValueError("No data variables found in dataset")
    return candidates[0]


def find_coord_names(ds):
    lat_names = ["lat", "latitude", "y"]
    lon_names = ["lon", "longitude", "x"]

    lat_name = next((c for c in lat_names if c in ds.coords), None)
    lon_name = next((c for c in lon_names if c in ds.coords), None)

    if lat_name is None or lon_name is None:
        raise ValueError(
            f"Could not find lat/lon coordinates in dataset. Found coords: {list(ds.coords)}"
        )
    return lat_name, lon_name


def gridcell_areas_m2(lat_1d, lon_1d):
    """Approximate geodesic cell areas for a regular lat/lon grid."""
    R = 6_371_000.0
    lat_rad = np.deg2rad(lat_1d)
    lon_rad = np.deg2rad(lon_1d)

    dlat = np.gradient(lat_rad)
    dlon = np.gradient(lon_rad)

    area = (R ** 2) * np.cos(lat_rad)[:, None] * dlat[:, None] * dlon[None, :]
    return np.abs(area)


def basin_timeseries(ds, basin_gdf, var_name=None):
    lat_name, lon_name = find_coord_names(ds)
    if var_name is None:
        var_name = pick_data_var(ds)

    da_var = ds[var_name]

    # Ensure dataset has standard dimension order: time, lat, lon
    time_name = "time" if "time" in da_var.dims else "valid_time"
    da_var = da_var.transpose(time_name, lat_name, lon_name)

    # Project basin geometry into geographic CRS for point-in-polygon test
    if basin_gdf.crs is None:
        basin_geo = basin_gdf.set_crs("EPSG:4326")
    else:
        basin_geo = basin_gdf.to_crs("EPSG:4326")

    basin_geom = basin_geo.union_all()

    lat = ds[lat_name].values
    lon = ds[lon_name].values
    lon2d, lat2d = np.meshgrid(lon, lat)

    points = gpd.GeoSeries(
        gpd.points_from_xy(lon2d.ravel(), lat2d.ravel()), crs="EPSG:4326"
    )
    inside = points.intersects(basin_geom).to_numpy().reshape(lat2d.shape)

    # Mask outside basin
    masked = da_var.where(inside)

    # Area-weighted basin mean (e.g., basin-average rainfall depth)
    area = gridcell_areas_m2(lat, lon)
    area_in = np.where(inside, area, np.nan)

    weights_da = xr.DataArray(
        area_in,
        coords={lat_name: ds[lat_name], lon_name: ds[lon_name]},
        dims=(lat_name, lon_name),
    )

    basin_mean = (masked * weights_da).sum(dim=(lat_name, lon_name), skipna=True) / weights_da.sum(
        dim=(lat_name, lon_name), skipna=True
    )

    # Basin-integrated volume if variable is depth in mm per timestep
    # (mm -> m, then multiply by cell area)
    basin_volume_m3 = ((masked / 1000.0) * weights_da).sum(
        dim=(lat_name, lon_name), skipna=True
    )

    out = pd.DataFrame(
        {
            "time": ds[time_name].values,
            f"{var_name}_basin_mean": basin_mean.values,
            f"{var_name}_basin_volume_m3": basin_volume_m3.values,
        }
    )
    return out, var_name


# ---- Paths: works in both local VS Code and Colab-style folder layouts ----
root = Path.cwd()
if (root / "data").exists():
    data_dir = root / "data"
elif (root / "rainfall_kf" / "data").exists():
    data_dir = root / "rainfall_kf" / "data"
else:
    raise FileNotFoundError("Could not find data directory from current working directory")

nc_path = data_dir / "gpm_sjv_huc8_subset.nc"
shp_path = data_dir / "HUC8 - CONUS Upper Merced.shp"

# Load data
ds = xr.open_dataset(nc_path)
basin = gpd.read_file(shp_path)

# Compute basin-bucket scalar time series
basin_df, used_var = basin_timeseries(ds, basin)
print(f"Using variable: {used_var}")
basin_df.head()
